List All Imports

In [1]:
import os
from chess import pgn
from tqdm import tqdm
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from chess import Board
import tensorflow as tf
import pickle
import json

Load raw games and preprocess them for data set

In [2]:
# get the game files
files = [file for file in os.listdir("simulated_games_filtered_PGN") if file.endswith(".pgn")]

In [3]:
# load the games
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
            
    return games

In [4]:
# write all the games together 

games = []
for file in tqdm(files):
    games.extend(load_pgn(f"simulated_games_filtered_PGN/{file}"))

100%|██████████| 11/11 [00:26<00:00,  2.41s/it]


In [5]:
len(games) # check how many 

10085

In [6]:
#translating the board into a matrix for the predition process 
def board_to_matrix(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix

# create the inputs based off game position and next move 
def create_input_for_nn(games):
    X = []
    y = []
    for game in games:
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return X, y

# encode all the moves
def encode_moves(moves):
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int

In [7]:
# create the training data
X, y = create_input_for_nn(games)
y, move_to_int = encode_moves(y)
y = tf.keras.utils.to_categorical(y, num_classes=len(move_to_int))
X = np.array(X)

Load Pretrained model, freeze convolutional layers and create new sequence of dense layers for transfer learning

In [8]:
pretrained_model = tf.keras.models.load_model("simulated_filtered_model(original)/SSMF_50EPOCHS.keras")

print("Layers in pretrained model:")
for i, layer in enumerate(pretrained_model.layers):
    print(f"{i}: {layer.name}")

conv_base = tf.keras.Sequential(pretrained_model.layers[:3])

conv_base.build((None, 8, 8, 12)) 

conv_base.trainable = False

new_model = tf.keras.Sequential([
        conv_base,
        tf.keras.layers.Dense(2048, activation='relu', name='new_dense1'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1024, activation='relu', name='new_dense2'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(len(move_to_int), activation='softmax', name='new_output')
    ])

Layers in pretrained model:
0: conv2d
1: conv2d_1
2: flatten
3: dense
4: dense_1


Compile and train new layers in model

In [ ]:
new_model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
new_model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

new_model.fit(X, y, epochs=50, validation_split=0.1, batch_size=64, callbacks=callbacks)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 2048)           │        80,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_dense1 (Dense)              │ (None, 2048)           │     4,196,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_dense2 (Dense)              │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_output (Dense)              │ (None, 1945)           │     1,993,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,368,985 (31.93 MB)

 Trainable params: 8,288,153 (31.62 MB)

 Non-trainable params: 80,832 (315.75 KB)

Epoch 1/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 220s 14ms/step - accuracy: 0.0549 - loss: 5.7039 - val_accuracy: 0.0808 - val_loss: 4.8647
Epoch 2/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 220s 14ms/step - accuracy: 0.0812 - loss: 4.7652 - val_accuracy: 0.0929 - val_loss: 4.4754
Epoch 3/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 228s 14ms/step - accuracy: 0.0920 - loss: 4.4918 - val_accuracy: 0.1001 - val_loss: 4.3258
Epoch 4/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 221s 14ms/step - accuracy: 0.0997 - loss: 4.3448 - val_accuracy: 0.1023 - val_loss: 4.2371
Epoch 5/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 211s 13ms/step - accuracy: 0.1060 - loss: 4.2460 - val_accuracy: 0.1057 - val_loss: 4.1867
Epoch 6/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 223s 14ms/step - accuracy: 0.1110 - loss: 4.1727 - val_accuracy: 0.1082 - val_loss: 4.1479
Epoch 7/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 217s 14ms/step - accuracy: 0.1154 - loss: 4.1121 - val_accuracy: 0.1083 - val_loss: 4.1221
Epoch 8/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 217s 14ms/s

Save model and related data to training and configuration

In [10]:
# save the model
new_model.save("simulated_filtered_modelV2(transfer_learning_original)/SSMF_50EPOCHS.keras")

# save the encoding
with open("simulated_filtered_modelV2(transfer_learning_original)/move_to_int.pkl", "wb") as f:
    pickle.dump(move_to_int, f)
int_to_move = {v: k for k, v in move_to_int.items()}
with open("simulated_filtered_modelV2(transfer_learning_original)/int_to_move.pkl", "wb") as f:
    pickle.dump(int_to_move, f)
# configuration 
config = {
    "epochs": 50,
    "batch_size": 64 ,
    "validation_split": 0.1,
    "optimizer": "Adam",
    "input_shape": (8, 8, 12),
}
# save the configurations
with open("simulated_filtered_model(transfer_learning_original)/train_config.json", "w") as f:
    json.dump(config, f, indent=4)